## Status (created 2026-08-04) -- READ BEFORE RUNNING

**Standalone notebook: Harmony at its conventional 50-PC setting, plus scProto at a
matched 50-dim latent.** Deliberately separate from
`batch_correct_then_cluster_baselines.ipynb` (Harmony at d=8) and
`combat_then_cluster_baselines.ipynb`, so all three can run in parallel Colab sessions
without stomping each other's config/state. Same shared helper module
(`interpretable_ssl/evaluation/batch_correct_baselines.py`) -- the only difference is
`force_matched_n_comps=50`.

**Nothing here overwrites the existing d=8 results.** The dimension is encoded in every
tag/folder/cache name (`X_harmony_d50`, `seacell_X_harmony_d50`,
`leiden_X_harmony_d50_K{K}`), and scProto's 50-dim run gets its own `LD50` folder --
so the d=8 runs (including the canonical scProto runs whose numbers match the paper) are
read-only here and appear in the tables below alongside the new d=50 rows.

**Everything is load-if-exists.** `SKIP_IF_EXISTS = True` makes a re-run of any Harmony
cell a metrics/cache reload rather than a recompute; the scProto cells auto-detect an
existing `LD50` checkpoint and reload it instead of retraining. Safe to re-run any cell
after a Colab disconnect.

# Harmony at the conventional 50-PC setting + scProto at a matched 50-dim latent

**Purpose.** Reviewer nG29, round 3: *"running Harmony at 8 PCs to match scProto's latent
dimensionality is questionable in my opinion. Harmony is conventionally used at ~50 PCs,
so restricting it to 8 handicaps the comparison in the authors' favour; I would
characterise the original 50-dim run as the fairer comparison rather than a
dimensionality bug. Reporting both dimensionalities would remove the concern entirely,
whichever way it resolves."*

This notebook produces exactly that second dimensionality, from both directions:

1. **Harmony at d=50** -- PCA to 50 components, then `harmonypy.run_harmony`, then
   SEACells and Leiden on top, identical downstream pipeline to every other baseline.
2. **scProto at latent_dims=50** -- the same scProto configuration as the paper's
   canonical runs, trained with a 50-dim latent instead of 8, so the comparison is
   matched at 50 as well as at 8.

Both are reported with the paper's own rare-cell metrics (coverage, homogeneity,
cross-batch homogeneity, macro F1 -- mean +- std across batches) plus the paired
one-sided Wilcoxon significance tests, next to the existing d=8 rows.

**On what "Harmony's default dimension" actually is.** Harmony does not choose one:
`harmonypy.run_harmony` performs no PCA at all -- it takes an embedding the caller
supplies (its `data_mat` argument is documented as a PCA embedding), and its only
dimension-adjacent default is the internal soft-cluster count `nclust = min(N/30, 100)`.
The 50 used here is the surrounding ecosystem's convention: scanpy's `sc.pp.pca` and
Seurat's `RunPCA` both default to 50 components, and `RunHarmony` consumes all
components of the reduction it is given. (The harmony R package's own legacy
`HarmonyMatrix(do_pca=TRUE, npcs=20)` entry point, and the authors' quickstart vignette,
use 20.) So d=50 is the reviewer's stated convention, which is what this notebook runs.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 254.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 224.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 226.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 261.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 152.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.2/711.2 kB 84.9 MB/s eta 0:00:00
   ━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 192.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 whi

In [1]:
# IMPORTANT: restart the runtime after this cell before running the cells below --
# numpy/scipy/anndata are C-extension linked, an in-process upgrade alone won't
# reliably take effect on already-imported modules.

In [2]:
# Ground truth for "did the install cell above actually work" -- pip's own log is
# noisy (resolver backtracking prints "Getting requirements to build wheel" errors
# for discarded candidate versions even on a fully successful install), so eyeballing
# it is unreliable. Actually importing every package we just installed is the real test.
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'umap-learn': 'umap', 'harmonypy': 'harmonypy',
    'faiss-cpu': 'faiss',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        ver = getattr(mod, '__version__', '?')
        print(f"  OK   {pkg_name:16s} (import {import_name}, version {ver})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s} (import {import_name}): {type(e).__name__}: {e}")

if _failed:
    print(f"\n{len(_failed)} package(s) failed to import: {_failed} -- re-run that "
          f"package's specific pip install line above and check its full error "
          f"output before proceeding.")
else:
    print(f"\nAll {len(_checks)} packages import cleanly -- safe to continue.")

  OK   numpy            (import numpy, version 2.2.6)
  OK   scipy            (import scipy, version 1.13.1)


/tmp/ipykernel_2483/3725702889.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  ver = getattr(mod, '__version__', '?')
/tmp/ipykernel_2483/3725702889.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  ver = getattr(mod, '__version__', '?')


  OK   anndata          (import anndata, version 0.13.2)
  OK   scanpy           (import scanpy, version 1.12.3)


  FAIL scarches         (import scarches): ImportError: cannot import name 'read' from 'anndata' (/usr/local/lib/python3.12/dist-packages/anndata/__init__.py)
  OK   scvi-tools       (import scvi, version 1.5.0.post1)
  OK   seacells         (import SEACells, version 0.3.3)
  OK   palantir         (import palantir, version 1.4.5)
  OK   scib-metrics     (import scib_metrics, version 0.6.0)
  OK   leidenalg        (import leidenalg, version 0.12.0)
  OK   python-igraph    (import igraph, version 1.0.0)
  OK   umap-learn       (import umap, version 0.5.12)
  OK   harmonypy        (import harmonypy, version 2.0.0)
  OK   faiss-cpu        (import faiss, version 1.14.1)

1 package(s) failed to import: ['scarches'] -- re-run that package's specific pip install line above and check its full error output before proceeding.


In [3]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [4]:
# Extra imports not already covered by nb_setup.py (which already pulls in run_mc_task,
# load_task1_multi, show_table, rare_celltype_purity_table, extract_model_key,
# TASK1_METRICS, TASK2_METRICS via `from interpretable_ssl.evaluation.paper_figures
# import *` and `from ...metric_helpers.result_tables import *`).
import os, glob
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import get_dataset_model_dir

print("extra imports ready")

extra imports ready


## Config

`CVAE_EPOCHS=50` / `BATCH_SIZE=1024` are the same Stage-1 hyperparameters
`train_scproto.ipynb` used for the published checkpoints -- kept identical so the only
thing that differs between the canonical scProto run and the one trained here is
`latent_dims`.

`TRAIN_EPOCHS=20` is a deliberately short Stage-2 budget (early stopping on modularity
usually halts before the cap anyway). The published d=8 runs used a 50-epoch cap with
the same early stopping -- raise this to 50 if the d=50 result lands close enough to the
d=8 one that the training budget could be questioned.

In [5]:
RNA_SEQ_DATASETS = ['pancreas']   # start with pancreas only; lung + immune have their
                                   # own run cells further down, and the final section
                                   # re-renders every table across all three.

HARMONY_DIM = 50        # PCA components fed to harmonypy -- the scanpy/Seurat
                        # ecosystem default, i.e. the setting Reviewer nG29 asked for.
SCPROTO_LATENT_DIM = 50 # scProto's latent_dims for the matched run (published: 8).

# scProto Stage-1 / Stage-2 training budget -- see Config markdown above.
CVAE_EPOCHS = 50
TRAIN_EPOCHS = 20
BATCH_SIZE = 1024
EVAL_FREQ = 3
PATIENCE = 6
UMAP_STEPS_PER_EPOCH = 500

SKIP_IF_EXISTS = True   # Harmony side: a completed (embedding + SEACells + Leiden) run
                        # for this dataset/dimension is reloaded, never recomputed.

CORRECTION_METHODS = ['harmony']
METHOD_DISPLAY_NAMES = {'harmony': f'Harmony d={HARMONY_DIM}'}

RUN_SEACELLS = True
RUN_LEIDEN = True

dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}
ALL_RNA_SEQ_DATASETS = ['pancreas', 'lung', 'pbmc-immune']

# Display name of the matched-dimension scProto row -- used as the reference in the
# same-dimension significance tests further down. Defined here (not next to its first
# use) so the all-datasets section at the bottom can be run on its own after a restart.
REF_D50 = f'scProto (d={SCPROTO_LATENT_DIM})'

## Helper functions

Both halves reuse existing code: Harmony goes through the same
`run_all_baselines_for_dataset` every other baseline notebook calls (with
`force_matched_n_comps=HARMONY_DIM`, the one parameter added for this notebook -- it
overrides the default behavior of deriving the dimension from scProto's own latent), and
scProto goes through `run_mc_task`, the same entry point that produced the paper's
numbers, with `latent_dims` overridden.

The thin wrappers below only add the load-if-exists detection for scProto's `LD50`
checkpoint and the resolution of its on-disk run key for the results tables.

In [6]:
from interpretable_ssl.evaluation.batch_correct_baselines import run_all_baselines_for_dataset
from interpretable_ssl.experiments.tasks import run_mc_task, LAMBDA_PROTO_UMAP_PRECON
from interpretable_ssl.evaluation.metric_helpers.result_tables import extract_model_key


def run_harmony_at_default_dim(ds_id):
    """Harmony at HARMONY_DIM PCs -> {SEACells, Leiden}, identical downstream pipeline
    to every other baseline. Writes to dimension-qualified folders
    (seacell_X_harmony_d50 / leiden_X_harmony_d50_K{K}), so the existing d=8 run is
    untouched. Re-running is a cache/metrics reload while SKIP_IF_EXISTS is True.
    """
    return run_all_baselines_for_dataset(
        ds_id, correction_methods=CORRECTION_METHODS, skip_if_exists=SKIP_IF_EXISTS,
        run_seacells=RUN_SEACELLS, run_leiden=RUN_LEIDEN,
        force_matched_n_comps=HARMONY_DIM,
    )


def scproto_run_dirs(ds_id, latent_dim=None):
    """Run folders for a scProto model trained at `latent_dim` for this dataset.
    latent_dims shows up in the folder name as 'LD{n}' whenever it differs from the
    default 8 (interpretable_ssl/model_name.py + constants.ABBREVIATIONS), which is what
    keeps this run separate from the published d=8 one on disk.
    """
    latent_dim = SCPROTO_LATENT_DIM if latent_dim is None else latent_dim
    base = get_dataset_model_dir(ds_id)
    if not os.path.isdir(base):
        return []
    token = f'LD{latent_dim}'
    return sorted(
        d for d in os.listdir(base)
        if token in d and d.startswith('proto_umap') and os.path.isdir(os.path.join(base, d))
    )


def scproto_checkpoint_exists(ds_id, latent_dim=None):
    """True if a Stage-2 checkpoint for this (dataset, latent_dim) is already saved --
    drives load-if-exists, so a Colab disconnect never costs a retrain. Deliberately NOT
    tasks._checkpoint_exists: that matches on experiment_name ('proto_umap') alone,
    which would also match the published d=8 run and wrongly report a hit here.
    """
    base = get_dataset_model_dir(ds_id)
    return any(
        os.path.exists(os.path.join(base, d, 'umap_checkpoint.pth'))
        for d in scproto_run_dirs(ds_id, latent_dim)
    )


def train_scproto_at_dim(ds_id, latent_dim=None):
    """scProto with latent_dims=latent_dim, everything else identical to the canonical
    published configuration (LAMBDA_PROTO_UMAP_PRECON, arbf affinity, dataset's own
    num_prototypes). Reloads the checkpoint if one already exists.
    """
    latent_dim = SCPROTO_LATENT_DIM if latent_dim is None else latent_dim
    load_umap = scproto_checkpoint_exists(ds_id, latent_dim)
    print(f"=== [{ds_id}] scProto latent_dims={latent_dim} "
          f"[{'reloading existing checkpoint' if load_umap else 'training fresh'}] ===")
    return run_mc_task(
        ds_id,
        cvae_epochs=CVAE_EPOCHS,
        train_epochs=TRAIN_EPOCHS,
        eval_freq=EVAL_FREQ,
        patience=PATIENCE,
        batch_size=BATCH_SIZE,
        umap_steps_per_epoch=UMAP_STEPS_PER_EPOCH,
        lambda_config=LAMBDA_PROTO_UMAP_PRECON,
        affinity_type='arbf',
        load_umap=load_umap,
        trainer_kwargs={'latent_dims': latent_dim},
    )


def scproto_model_key(ds_ids, latent_dim=None):
    """The single model key every scProto-at-latent_dim run normalizes to, for the
    results tables (extract_model_key strips the dataset/NP/cvae/version tokens, so all
    three datasets share one key -- one row spanning every dataset). Returns None if
    nothing has been trained yet, and warns if the runs on disk disagree.
    """
    keys = {
        extract_model_key(d, ds_id=ds)
        for ds in ds_ids for d in scproto_run_dirs(ds, latent_dim)
    }
    if not keys:
        print(f"No scProto LD{latent_dim or SCPROTO_LATENT_DIM} run found on disk yet "
              f"for {ds_ids} -- run the training cell(s) first.")
        return None
    if len(keys) > 1:
        print(f"WARNING: multiple distinct scProto keys on disk: {sorted(keys)} -- "
              f"using '{sorted(keys)[0]}'. Check for a stale run from an earlier config.")
    return sorted(keys)[0]


print("helpers ready")

helpers ready


## Pancreas -- Harmony at d=50

SEACells and Leiden both run on the 50-PC Harmony embedding, at K = the dataset's
`num_prototypes` (220), matching the paper's protocol that every baseline produces the
same number of metacells as scProto.

In [7]:
pancreas_harmony = run_harmony_at_default_dim('pancreas')

loading pancreas data
✅ Already subsetted to HVGs (4000 genes).


 captum (see https://github.com/pytorch/captum).


dataset is None, loading pancreas
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [9]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.446/23.916/92.201, effk_med=62.4, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pancreas/pretrain/pretrain_ds-pancreas_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pancreas', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'tech'}
📊 EdgeDataset: 1155146 edges
   Weight range: [0.0320, 0.9688]
   umap_steps_per_epoch=500 → 512000 edges/epoch (of 1155146 total

  0%|          | 0/16 [00:00<?, ?it/s]

[pancreas] force_matched_n_comps=50 overrides scProto's own latent dimension (8) for ['harmony'].

=== [pancreas] batch-correction method: harmony ===


...

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/16382 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/16382 [00:00<?, ?it/s]

Constructing CSR matrix...
[pancreas] scProto (d=8): 0.454 +/- 0.184 (n=8 batches)
[pancreas] affinity purity saved to /content/drive/MyDrive/models/pancreas/rare_affinity_purity_pancreas.json (merged with any existing entries from other runs/notebooks)


## Pancreas -- scProto at latent_dims=50

Same scProto configuration as the published runs, latent dimension changed to 50 so the
comparison against Harmony is matched at 50 as well.

In [8]:
t_panc, res_panc, mc_panc = train_scproto_at_dim('pancreas')
print(f"\nrun dir: {t_panc.get_dump_path()}")
res_panc

=== [pancreas] scProto latent_dims=50 [training fresh] ===
dataset is None, loading pancreas
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-panc_NP220_LD50_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [9]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 50
Decoder Architecture:
	First Layer in, out and cond:  50 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.446/23.916/92.201, effk_med=62.4, mutual=100.00%
adam
The model is being trained without using prototypes.


INFO:scarches.trainers.scpoli.trainer:GPU available: True, GPU used: True


Initializing dataloaders
Starting training
 |████████████████████| 100.0%  - val_loss: 2299.26 - val_cvae_loss: 2299.26
[waypoint init] N=16382  K=220  n_eigs=10  nnz=1155146  nnz/row=70.5  w[min/mean/max]=3.203e-02/3.392e-01/9.688e-01  deg[min/mean/max]=5.45/23.92/92.20
[waypoint init] computing diffusion map ...
[waypoint init] diffusion map done — 10 eigenvectors


waypoint MaxMin: 100%|██████████| 219/219 [00:00<00:00, 9207.81proto/s]

[waypoint init] selected 220 seed cells


...

Saved metacells (220 prototypes) to /content/drive/MyDrive/models//pancreas/proto_umap_ds-panc_NP220_LD50_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31/metacells.h5ad


  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

UMAP data saved to /content/drive/MyDrive/models//pancreas/proto_umap_ds-panc_NP220_LD50_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31

run dir: /content/drive/MyDrive/models//pancreas/proto_umap_ds-panc_NP220_LD50_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31


{'seed': 31,
 'purity': 0.9162471677544743,
 'niche_purity': None,
 'batch_entropy': 0.36533180735181503,
 'modularity': 0.6870671426432393,
 'coverage': 0.9285714285714286,
 'dge_rbo_avg': 0.20866837565536187,
 'dge_kendall_avg': 0.21393399219553458,
 'dge_jaccard_avg': 0.24838556195260025,
 'scgraph_corr_avg': 0.904885004724838,
 'scgraph_corr_std': 0.039289894316753116,
 'ct_niche_rbo_avg': None,
 'aff_compactness_per_batch': {'celseq': 0.1544415803379006,
  'celseq2': 0.04000456534238699,
  'fluidigmc1': 0.2910346428403924,
  'inDrop1': 0.06453792141598644,
  'inDrop2': 0.04967949449132574,
  'inDrop3': 0.1576306696360339,
  'inDrop4': 0.03784295482164924,
  'smarter': 0.2256385805168199,
  'smartseq2': 0.20196440778604788},
 'aff_compactness_mean': 0.5523880968396582}

## Pancreas -- results

Every table below is scored the same way as the paper's own: Table 1 (modularity, batch
entropy, purity), Table 2 (coverage, scGraph), and the rare-cell table (coverage,
recall, precision, homogeneity, cross-batch homogeneity, macro F1 -- mean +- std across
batches), followed by the paired one-sided Wilcoxon tests, Bonferroni-corrected per
dataset.

Four rows carry the comparison: **scProto** (published, d=8), **scProto (d=50)**,
**Harmony d=50** (this notebook), and **Harmony d=8** (read from the existing run --
never recomputed here). SEACells (PCA) comes along as the paper's own baseline.

In [9]:
from interpretable_ssl.evaluation.rebuttal_report import (
    build_model_keywords, dim_matched_read_only_keywords, render_full_comparison_report,
    RARE_CELL_SIG_METRICS,
)

# Harmony d=50 (computed here) + Harmony d=8 (read-only, from the other notebook's run).
# Distinct display names on purpose -- identical names would collapse into one row and
# silently drop one of the two dimensionalities the reviewer asked to see side by side.
MODEL_KEYWORDS = build_model_keywords(
    CORRECTION_METHODS, METHOD_DISPLAY_NAMES, matched_dim=HARMONY_DIM,
    extra_read_only=dim_matched_read_only_keywords('harmony', 'Harmony d=8', matched_dim=8),
)

SCPROTO_D50_KEY = scproto_model_key(RNA_SEQ_DATASETS)
if SCPROTO_D50_KEY:
    MODEL_KEYWORDS[SCPROTO_D50_KEY] = f'scProto (d={SCPROTO_LATENT_DIM})'

MODEL_KEYWORDS

{'proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp': 'scProto',
 'seacell': 'SEACells (PCA)',
 'seacell_X_harmony_d50': 'SEACells (Harmony d=50)',
 'leiden_X_harmony_d50': 'Leiden (Harmony d=50)',
 'seacell_X_harmony_d8': 'SEACells (Harmony d=8)',
 'leiden_X_harmony_d8': 'Leiden (Harmony d=8)',
 'proto_umap_LD50_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp': 'scProto (d=50)'}

In [10]:
# scProto (published, d=8) as reference: the headline question -- does the paper's model
# still hold up against Harmony run at the reviewer's conventional 50-PC setting?
report_panc = render_full_comparison_report(
    RNA_SEQ_DATASETS, dataset_display_names, MODEL_KEYWORDS, ref_name='scProto',
)

=== Table 1: community structure / batch integration ===



=== Table 2: metacell representation quality ===


  [scProto|pancreas] resolving run dir ...  [SEACells (PCA)|pancreas] resolving run dir ...

  [SEACells (Harmony d=50)|pancreas] resolving run dir ...
  [Leiden (Harmony d=50)|pancreas] resolving run dir ...
  [SEACells (Harmony d=8)|pancreas] resolving run dir ...
  [Leiden (Harmony d=8)|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] run dir resolved (0.0s)
  [scProto (d=50)|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] reading umap_cells.csv ...
  [scProto|pancreas] run dir resolved (0.0s)  [SEACells (Harmony d=50)|pancreas] run dir resolved (0.0s)

  [SEACells (Harmony d=8)|pancreas] run dir resolved (0.0s)
  [Leiden (Harmony d=8)|pancreas] run dir resolved (0.0s)
  [Leiden (Harmony d=50)|pancreas] run dir resolved (0.0s)
  [scProto (d=50)|pancreas] run dir resolved (0.0s)
  [Leiden (Harmony d=8)|pancreas] reading umap_cells.csv ...
  [scProto|pancreas] reading umap_cells.csv ...
  [SEACells (Harmony d=50)|pancreas] reading umap_cells.csv ...  [SEACells (


=== UNPAIRED rare-cell significance (Mann-Whitney U) ===


,dataset,metric,method,k,n,median,mean,std,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,scProto,219,8,0.437680,0.535403,0.207961,NaN,NaN,NaN
1,Pancreas,batch_rare_f1_macro,SEACells (PCA),220,8,0.372628,0.370589,0.250869,0.032643,0.195857,ns
2,Pancreas,batch_rare_f1_macro,SEACells (Harmony d=50),220,8,0.597360,0.658295,0.130668,0.958892,1.000000,ns
3,Pancreas,batch_rare_f1_macro,Leiden (Harmony d=50),220,8,0.368298,0.414804,0.218054,0.080264,0.481585,ns
4,Pancreas,batch_rare_f1_macro,SEACells (Harmony d=8),220,8,0.295521,0.316501,0.159421,0.014064,0.084382,ns
5,Pancreas,batch_rare_f1_macro,Leiden (Harmony d=8),220,8,0.133361,0.223580,0.171156,0.002331,0.013986,*
6,Pancreas,batch_rare_f1_macro,scProto (d=50),220,8,0.637381,0.602200,0.180641,0.779099,1.000000,ns
7,Pancreas,batch_rare_homogeneity,scProto,219,8,0.522129,0.564854,0.155263,NaN,NaN,NaN
8,Pancreas,batch_rare_homogeneity,SEACells (PCA),220,8,0.369265,0.420592,0.195138,0.010334,0.062005,ns
9,Pancreas,batch_rare_homogeneity,SEACells (Harmony d=50),220,8,0.591839,0.616825,0.146558,0.747319,1.000000,ns


...

dataset,Pancreas
method,
Leiden (Harmony d=50),"1.000 (K=220, n=220, wins=?/220) ns p_adj=1"
Leiden (Harmony d=8),"1.000 (K=220, n=220, wins=?/220) ns p_adj=1"
SEACells (Harmony d=50),"0.992 (K=220, n=220, wins=?/220) ns p_adj=1"
SEACells (Harmony d=8),"0.967 (K=220, n=220, wins=?/220) *** p_adj=0..."
SEACells (PCA),"0.993 (K=220, n=220, wins=?/220) ns p_adj=1"
scProto,"1.000 (K=219, n=219) [ref]"
scProto (d=50),"1.000 (K=220, n=220, wins=?/220) ns p_adj=1"


=== batch_entropy_per_mc: scProto vs. each same-K baseline, one-sided Mann-Whitney U (scProto > other), Bonferroni-corrected per dataset ===


dataset,Pancreas
method,
Leiden (Harmony d=50),"1.064 (K=220, n=220, wins=?/220) ns p_adj=1"
Leiden (Harmony d=8),"1.327 (K=220, n=220, wins=?/220) ns p_adj=1"
SEACells (Harmony d=50),"1.342 (K=220, n=220, wins=?/220) ns p_adj=1"
SEACells (Harmony d=8),"1.339 (K=220, n=220, wins=?/220) ns p_adj=1"
SEACells (PCA),"-0.000 (K=220, n=220, wins=?/220) *** p_adj=..."
scProto,"0.214 (K=219, n=219) [ref]"
scProto (d=50),"-0.000 (K=220, n=220, wins=?/220) ns p_adj=1"



=== PAIRED Table 1 significance (modularity only, recommended for that metric) ===
=== modularity_per_batch: scProto vs. each same-K baseline, PAIRED one-sided Wilcoxon signed-rank (scProto > other), Bonferroni-corrected per dataset ===


dataset,Pancreas
method,
Leiden (Harmony d=50),"0.532 (K=220, n=9, wins=9.0/9) * p_adj=0.0117"
Leiden (Harmony d=8),"0.614 (K=220, n=9, wins=6.0/9) ns p_adj=1"
SEACells (Harmony d=50),"0.419 (K=220, n=9, wins=9.0/9) * p_adj=0.0117"
SEACells (Harmony d=8),"0.566 (K=220, n=9, wins=7.0/9) ns p_adj=0.75"
SEACells (PCA),"0.658 (K=220, n=9, wins=0.0/9) ns p_adj=1"
scProto,"0.621 (K=219, n=9) [ref]"
scProto (d=50),"0.652 (K=220, n=9, wins=3.0/9) ns p_adj=1"


In [11]:
# Same-dimension comparison: scProto (d=50) as reference, against Harmony d=50.
# Reuses the rare table already computed above -- no recompute, no retraining.
from interpretable_ssl.evaluation.paper_figures import (
    rare_metric_significance_paired, graph_batch_significance_paired,
)

sig_rare_d50 = rare_metric_significance_paired(
    report_panc['rare'], ref_name=REF_D50, metrics=RARE_CELL_SIG_METRICS,
    dataset_display_names=dataset_display_names,
)
print(f"=== Rare-cell metrics, PAIRED one-sided Wilcoxon ({REF_D50} > other), "
      f"Bonferroni-corrected per dataset ===")
display(sig_rare_d50)

sig_mod_d50 = graph_batch_significance_paired(
    RNA_SEQ_DATASETS, MODEL_KEYWORDS, ref_name=REF_D50,
    dataset_display_names=dataset_display_names,
)
print(f"\n=== Modularity per batch, PAIRED one-sided Wilcoxon ({REF_D50} > other) ===")
display(sig_mod_d50)

=== Rare-cell metrics, PAIRED one-sided Wilcoxon (scProto (d=50) > other), Bonferroni-corrected per dataset ===


,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,scProto,219,8,0.437680,0.535403,0.207961,5.0,0.156250,0.937500,ns
1,Pancreas,batch_rare_f1_macro,SEACells (PCA),220,8,0.372628,0.370589,0.250869,7.0,0.008980,0.053881,ns
2,Pancreas,batch_rare_f1_macro,SEACells (Harmony d=50),220,8,0.597360,0.658295,0.130668,3.0,0.769531,1.000000,ns
3,Pancreas,batch_rare_f1_macro,Leiden (Harmony d=50),220,8,0.368298,0.414804,0.218054,8.0,0.003906,0.023438,*
4,Pancreas,batch_rare_f1_macro,SEACells (Harmony d=8),220,8,0.295521,0.316501,0.159421,8.0,0.003906,0.023438,*
5,Pancreas,batch_rare_f1_macro,Leiden (Harmony d=8),220,8,0.133361,0.223580,0.171156,8.0,0.003906,0.023438,*
6,Pancreas,batch_rare_f1_macro,scProto (d=50),220,8,0.637381,0.602200,0.180641,NaN,NaN,NaN,NaN
7,Pancreas,batch_rare_homogeneity,scProto,219,8,0.522129,0.564854,0.155263,6.0,0.156250,0.937500,ns
8,Pancreas,batch_rare_homogeneity,SEACells (PCA),220,8,0.369265,0.420592,0.195138,8.0,0.003906,0.023438,*
9,Pancreas,batch_rare_homogeneity,SEACells (Harmony d=50),220,8,0.591839,0.616825,0.146558,4.0,0.679688,1.000000,ns



=== Modularity per batch, PAIRED one-sided Wilcoxon (scProto (d=50) > other) ===


,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,modularity_per_batch,scProto,219,9,0.621456,0.601234,0.083385,6.0,0.101562,0.609375,ns
1,Pancreas,modularity_per_batch,SEACells (PCA),220,9,0.657892,0.673906,0.050910,1.0,0.996094,1.000000,ns
2,Pancreas,modularity_per_batch,SEACells (Harmony d=50),220,9,0.419409,0.431852,0.036186,9.0,0.001953,0.011719,*
3,Pancreas,modularity_per_batch,Leiden (Harmony d=50),220,9,0.531719,0.501579,0.122383,9.0,0.001953,0.011719,*
4,Pancreas,modularity_per_batch,SEACells (Harmony d=8),220,9,0.566367,0.567150,0.017709,7.0,0.064453,0.386719,ns
5,Pancreas,modularity_per_batch,Leiden (Harmony d=8),220,9,0.614318,0.612469,0.021231,6.0,0.410156,1.000000,ns
6,Pancreas,modularity_per_batch,scProto (d=50),220,9,0.651824,0.610766,0.077252,NaN,NaN,NaN,NaN


### Pancreas -- realized cluster/metacell count vs. K

Confirms Harmony's downstream SEACells/Leiden landed at scProto's own `num_prototypes`,
so the comparison is at equal K. Reads saved outputs only -- no recompute.

In [12]:
from interpretable_ssl.evaluation.batch_correct_baselines import get_realized_seacell_count

target_k = {ds: DATASETS[ds]['num_prototypes'] for ds in RNA_SEQ_DATASETS}
harmony_tag = f'X_harmony_d{HARMONY_DIM}'

df_k = load_task1_multi(RNA_SEQ_DATASETS, metrics=['n_clusters', 'resolution'])
if df_k.empty:
    print("No runs found yet under MODEL_DIR -- run the cells above first.")
else:
    is_leiden = df_k.index.get_level_values('run').str.startswith(f'leiden_{harmony_tag}')
    df_k_leiden = df_k[is_leiden].copy()
    if df_k_leiden.empty:
        print(f"No leiden_{harmony_tag} run found yet.")
    else:
        df_k_leiden['target_k'] = [target_k[ds] for ds, _run in df_k_leiden.index]
        df_k_leiden['matches_target'] = df_k_leiden['n_clusters'] == df_k_leiden['target_k']
        display(df_k_leiden)

seacell_k_rows = [{
    'dataset': ds_id, 'method': harmony_tag,
    'n_actual': get_realized_seacell_count(ds_id, harmony_tag),
    'target_k': target_k[ds_id],
} for ds_id in RNA_SEQ_DATASETS]
df_seacell_k = pd.DataFrame(seacell_k_rows)
df_seacell_k['matches_target'] = df_seacell_k.apply(
    lambda r: r['n_actual'] is not None and abs(r['n_actual'] - r['target_k']) <= 0.05 * r['target_k'],
    axis=1,
)
display(df_seacell_k.set_index(['dataset', 'method']))

,,n_clusters,resolution,target_k,matches_target
dataset,run,,,,
pancreas,leiden_X_harmony_d50_K220,220.0,32.0,220,True


,,n_actual,target_k,matches_target
dataset,method,,,
pancreas,X_harmony_d50,220,220,True


### Pancreas -- embedding-only rare-cell affinity purity (no clustering)

Isolates the embedding from whichever clustering runs on top: one ARBF affinity graph
built directly on each embedding, then for each locally-rare-type cell, the fraction of
its total affinity mass going to same-type cells. Ran automatically inside the Harmony
cell above (`compute_and_save_embedding_affinity_purity`); this just loads the saved
results. The `dim` column distinguishes the d=50 Harmony row from the d=8 one.

In [13]:
from interpretable_ssl.evaluation.batch_correct_baselines import load_and_compare_affinity_purity

df_affinity_purity = load_and_compare_affinity_purity(
    RNA_SEQ_DATASETS, dataset_display_names=dataset_display_names,
)
df_affinity_purity

,dataset,method,dim,n,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,Raw PCA (uncorrected),50,8,0.385,0.164,7.0,0.0742,0.4453,ns
1,Pancreas,scProto,8,8,0.454,0.184,NaN,NaN,NaN,NaN
2,Pancreas,scVI (Gaussian),8,8,0.261,0.267,7.0,0.0078,0.0469,*
3,Pancreas,Harmony,8,8,0.297,0.129,7.0,0.0195,0.1172,ns
4,Pancreas,scPoli (Stage-1),8,8,0.512,0.162,3.0,0.8750,1.0000,ns
5,Pancreas,scVI,8,8,0.365,0.175,8.0,0.0039,0.0234,*
6,Pancreas,Harmony,50,8,0.529,0.176,4.0,0.8086,1.0000,ns


## Lung and Immune

Same two runs per dataset (Harmony at d=50, scProto at latent_dims=50). Each cell is
independent and load-if-exists, so they can be run in any order, resumed after a
disconnect, or split across sessions.

### Lung -- Harmony at d=50

In [ ]:
lung_harmony = run_harmony_at_default_dim('lung')

loading lung data
✅ Already subsetted to HVGs (4000 genes).
dataset is None, loading lung
loading lung data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-lung_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [16]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.255/24.467/119.300, effk_med=63.8, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/lung/pretrain/pretrain_ds-lung_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'lung', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'batch'}
📊 EdgeDataset: 2447924 edges
   Weight range: [0.0137, 0.9491]
   umap_steps_per_epoch=500 →

  0%|          | 0/32 [00:00<?, ?it/s]

[lung] force_matched_n_comps=50 overrides scProto's own latent dimension (8) for ['harmony'].

=== [lung] batch-correction method: harmony ===


2026-08-04 07:41:56,355 - harmonypy - INFO - Running Harmony
INFO:harmonypy:Running Harmony
2026-08-04 07:41:56,356 - harmonypy - INFO -   Parameters:
INFO:harmonypy:  Parameters:
2026-08-04 07:41:56,356 - harmonypy - INFO -     max_iter_harmony: 10
INFO:harmonypy:    max_iter_harmony: 10
2026-08-04 07:41:56,356 - harmonypy - INFO -     max_iter_kmeans: 4
INFO:harmonypy:    max_iter_kmeans: 4
2026-08-04 07:41:56,357 - harmonypy - INFO -     epsilon_cluster: 0.001
INFO:harmonypy:    epsilon_cluster: 0.001
2026-08-04 07:41:56,357 - harmonypy - INFO -     epsilon_harmony: 0.01
INFO:harmonypy:    epsilon_harmony: 0.01
2026-08-04 07:41:56,357 - harmonypy - INFO -     nclust: 100
INFO:harmonypy:    nclust: 100
2026-08-04 07:41:56,358 - harmonypy - INFO -     block_size: 0.05
INFO:harmonypy:    block_size: 0.05
2026-08-04 07:41:56,358 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
INFO:harmonypy:    lamb: dynamic (alpha=0.2)
2026-08-04 07:41:56,358 - harmonypy - INFO -     theta: [2. 2. 2

[lung] harmony_d50_k100: cached embedding to /content/drive/MyDrive/models/lung/_cache_X_harmony_d50_k100_emb.npy
Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


...

  0%|          | 0/32472 [00:00<?, ?it/s]

Constructing CSR matrix...
[waypoint init] N=32472  k=300  n_eigs=10  nnz=2711740  nnz/row=83.5
[waypoint init] computing diffusion map ...
[waypoint init] diffusion map done — 10 eigenvectors


waypoint MaxMin: 100%|██████████| 299/299 [00:00<00:00, 5246.48archetype/s]

[waypoint init] selected 300 archetype seed cells
[SEACells backend] GPU detected but n_cells=32472 > 30000 -- forcing CPU+sparse anyway (GPU path is dense-only and would need a ~32472x32472 dense matrix, likely OOM)
[SEACells backend] using our own optimized sparse-CPU SEACells (never materializes kernel_matrix @ kernel_matrix.T)
Welcome to SEACells!
Using provided list of initial archetypes
Randomly initialized A matrix.


Setting convergence threshold at 262.28773
Starting iteration 1.
Completed iteration 1.


### Lung -- scProto at latent_dims=50

In [ ]:
t_lung, res_lung, mc_lung = train_scproto_at_dim('lung')
print(f"\nrun dir: {t_lung.get_dump_path()}")
res_lung

### Immune (PBMC) -- Harmony at d=50

In [ ]:
immune_harmony = run_harmony_at_default_dim('pbmc-immune')

### Immune (PBMC) -- scProto at latent_dims=50

In [ ]:
t_imm, res_imm, mc_imm = train_scproto_at_dim('pbmc-immune')
print(f"\nrun dir: {t_imm.get_dump_path()}")
res_imm

## All three datasets -- final tables

Re-renders every table above across pancreas, lung, and immune together. This is the
section to read the reported numbers off: rare-cell metrics as mean +- std across
batches, with paired one-sided Wilcoxon significance against both references (scProto
d=8, the published model, and scProto d=50, the dimension-matched one).

In [ ]:
# Rebuild the keyword map over all three datasets (scProto's LD50 key is resolved from
# whatever is on disk, so this works whether the runs above happened in this session or
# an earlier one). Imports repeated here so this section runs standalone after a restart.
from interpretable_ssl.evaluation.rebuttal_report import (
    build_model_keywords, dim_matched_read_only_keywords, render_full_comparison_report,
    RARE_CELL_SIG_METRICS,
)
from interpretable_ssl.evaluation.paper_figures import (
    rare_metric_significance_paired, graph_batch_significance_paired,
)
from interpretable_ssl.evaluation.batch_correct_baselines import load_and_compare_affinity_purity

MODEL_KEYWORDS_ALL = build_model_keywords(
    CORRECTION_METHODS, METHOD_DISPLAY_NAMES, matched_dim=HARMONY_DIM,
    extra_read_only=dim_matched_read_only_keywords('harmony', 'Harmony d=8', matched_dim=8),
)
SCPROTO_D50_KEY = scproto_model_key(ALL_RNA_SEQ_DATASETS)
if SCPROTO_D50_KEY:
    MODEL_KEYWORDS_ALL[SCPROTO_D50_KEY] = f'scProto (d={SCPROTO_LATENT_DIM})'

MODEL_KEYWORDS_ALL

In [ ]:
report_all = render_full_comparison_report(
    ALL_RNA_SEQ_DATASETS, dataset_display_names, MODEL_KEYWORDS_ALL, ref_name='scProto',
)

In [ ]:
# Same-dimension reference (scProto d=50), all three datasets.
sig_rare_d50_all = rare_metric_significance_paired(
    report_all['rare'], ref_name=REF_D50, metrics=RARE_CELL_SIG_METRICS,
    dataset_display_names=dataset_display_names,
)
print(f"=== Rare-cell metrics, PAIRED one-sided Wilcoxon ({REF_D50} > other), "
      f"Bonferroni-corrected per dataset ===")
display(sig_rare_d50_all)

sig_mod_d50_all = graph_batch_significance_paired(
    ALL_RNA_SEQ_DATASETS, MODEL_KEYWORDS_ALL, ref_name=REF_D50,
    dataset_display_names=dataset_display_names,
)
print(f"\n=== Modularity per batch, PAIRED one-sided Wilcoxon ({REF_D50} > other) ===")
display(sig_mod_d50_all)

In [ ]:
# Embedding-only affinity purity across all three datasets (d=50 and d=8 rows side by
# side -- see the `dim` column).
load_and_compare_affinity_purity(
    ALL_RNA_SEQ_DATASETS, dataset_display_names=dataset_display_names,
)

In [ ]:
# Flat CSV of the rare-cell table for all three datasets -- convenient to paste numbers
# from when writing the reply.
save_path = '/content/drive/MyDrive/models/rare_metrics_harmony_d50.csv'
report_all['rare'].to_csv(save_path)
print(f"saved to {save_path}")
report_all['rare']